# AgentML Experiment Analysis

This notebook pulls data directly from MLflow and auto-generates visualizations
for analyzing experiment results.

## Configuration

Set the experiment name or run ID you want to analyze below.

In [ ]:
# ============================================================
# CONFIGURE THIS: Set the experiment name or specific run ID
# ============================================================
EXPERIMENT_NAME = "agentml_experiment"  # Change this to your experiment name
SPECIFIC_RUN_ID = None                   # Set to a run ID to analyze a single run, or None for all
TRACKING_URI = "./mlruns"               # MLflow tracking URI
TOP_N = 5                                # Number of top models to show in summary

## Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, mean_squared_error,
)
from sklearn.model_selection import learning_curve

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Connect to MLflow
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)

# Get experiment
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experiment '{EXPERIMENT_NAME}' not found. Available experiments: "
                     f"{[e.name for e in client.search_experiments()]}")

print(f"Experiment: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")

# Load all runs
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.val_score DESC"],
)

if not runs:
    raise ValueError("No runs found for this experiment. Run some experiments first!")

print(f"Total runs: {len(runs)}")

# Build a dataframe of all runs
run_data = []
for run in runs:
    row = {
        "run_id": run.info.run_id,
        "model_name": run.data.params.get("model_name", "unknown"),
        "val_score": run.data.metrics.get("val_score", None),
        "cv_mean": run.data.metrics.get("cv_mean", None),
        "cv_std": run.data.metrics.get("cv_std", None),
        "training_time": run.data.metrics.get("training_time", None),
        "agent_notes": run.data.params.get("agent_notes", ""),
        "start_time": pd.to_datetime(run.info.start_time, unit="ms"),
    }
    # Add all hyperparameters
    for k, v in run.data.params.items():
        if k not in ["model_name", "agent_notes"]:
            row[f"param_{k}"] = v
    run_data.append(row)

df_runs = pd.DataFrame(run_data)
df_runs = df_runs.sort_values("start_time").reset_index(drop=True)
df_runs["experiment_num"] = range(1, len(df_runs) + 1)

# Load processed data for additional analysis
try:
    with open("processed/data_splits.pkl", "rb") as f:
        data = pickle.load(f)
    task_type = data["metadata"]["task_type"]
    print(f"Task type: {task_type}")
except FileNotFoundError:
    data = None
    task_type = "unknown"
    print("Warning: processed/data_splits.pkl not found. Some plots will be skipped.")

## 1. Experiment Comparison

Metric score across all runs, showing progression over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: val_score per run
colors = sns.color_palette("viridis", len(df_runs))
bars = axes[0].bar(df_runs["experiment_num"], df_runs["val_score"], color=colors)
axes[0].set_xlabel("Experiment #")
axes[0].set_ylabel("Validation Score")
axes[0].set_title("Validation Score per Experiment")
axes[0].axhline(y=df_runs["val_score"].max(), color="red", linestyle="--",
                alpha=0.7, label=f"Best: {df_runs['val_score'].max():.4f}")
axes[0].legend()

# Line chart: progression over time
axes[1].plot(df_runs["experiment_num"], df_runs["val_score"],
             marker="o", linewidth=2, markersize=8, color="steelblue")
axes[1].fill_between(df_runs["experiment_num"],
                     df_runs["val_score"] - df_runs["cv_std"].fillna(0),
                     df_runs["val_score"] + df_runs["cv_std"].fillna(0),
                     alpha=0.2, color="steelblue")
axes[1].set_xlabel("Experiment #")
axes[1].set_ylabel("Validation Score")
axes[1].set_title("Metric Progression Over Experiments")

# Add model name annotations
for _, row in df_runs.iterrows():
    axes[1].annotate(row["model_name"], (row["experiment_num"], row["val_score"]),
                     textcoords="offset points", xytext=(0, 10),
                     fontsize=7, ha="center", rotation=30)

plt.tight_layout()
plt.show()

## 2. Feature Importance

Feature importance plot for the best tree-based model.

In [ ]:
# Find the best run with a tree-based model
tree_models = ["RandomForestClassifier", "RandomForestRegressor",
               "GradientBoostingClassifier", "GradientBoostingRegressor",
               "XGBClassifier", "XGBRegressor",
               "ExtraTreesClassifier", "ExtraTreesRegressor",
               "DecisionTreeClassifier", "DecisionTreeRegressor"]

best_tree_run = None
for _, row in df_runs.sort_values("val_score", ascending=False).iterrows():
    if row["model_name"] in tree_models:
        best_tree_run = row
        break

if best_tree_run is not None and data is not None:
    # Load the model from MLflow
    model_uri = f"runs:/{best_tree_run['run_id']}/model"
    try:
        model = mlflow.sklearn.load_model(model_uri)
        feature_names = data["feature_names"]

        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
            indices = np.argsort(importances)[::-1]

            # Show top 20 features
            n_features = min(20, len(feature_names))
            top_indices = indices[:n_features]

            fig, ax = plt.subplots(figsize=(10, max(6, n_features * 0.4)))
            ax.barh(range(n_features),
                    importances[top_indices][::-1],
                    color="steelblue")
            ax.set_yticks(range(n_features))
            ax.set_yticklabels([feature_names[i] for i in top_indices][::-1])
            ax.set_xlabel("Feature Importance")
            ax.set_title(f"Feature Importance ({best_tree_run['model_name']})")
            plt.tight_layout()
            plt.show()
        else:
            print(f"Model {best_tree_run['model_name']} doesn't have feature_importances_")
    except Exception as e:
        print(f"Could not load model: {e}")
else:
    print("No tree-based model found in experiments or processed data not available.")

## 3. Confusion Matrix / Residual Plot

Confusion matrix for classification or residual plot for regression.

In [ ]:
if data is not None:
    # Load the best model overall
    best_run = df_runs.sort_values("val_score", ascending=False).iloc[0]
    model_uri = f"runs:/{best_run['run_id']}/model"

    try:
        best_model = mlflow.sklearn.load_model(model_uri)
        X_val = data["X_val"]
        y_val = data["y_val"]
        y_pred = best_model.predict(X_val)

        if task_type == "classification":
            # Confusion Matrix
            fig, ax = plt.subplots(figsize=(8, 8))
            cm = confusion_matrix(y_val, y_pred)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)
            disp.plot(ax=ax, cmap="Blues")
            ax.set_title(f"Confusion Matrix ({best_run['model_name']})")
            plt.tight_layout()
            plt.show()
        else:
            # Residual Plot
            residuals = y_val - y_pred

            fig, axes = plt.subplots(1, 2, figsize=(14, 6))

            # Predicted vs Actual
            axes[0].scatter(y_val, y_pred, alpha=0.5, color="steelblue")
            min_val = min(y_val.min(), y_pred.min())
            max_val = max(y_val.max(), y_pred.max())
            axes[0].plot([min_val, max_val], [min_val, max_val],
                        "r--", linewidth=2, label="Perfect prediction")
            axes[0].set_xlabel("Actual")
            axes[0].set_ylabel("Predicted")
            axes[0].set_title(f"Predicted vs Actual ({best_run['model_name']})")
            axes[0].legend()

            # Residual distribution
            axes[1].hist(residuals, bins=30, color="steelblue", edgecolor="black", alpha=0.7)
            axes[1].axvline(x=0, color="red", linestyle="--", linewidth=2)
            axes[1].set_xlabel("Residual")
            axes[1].set_ylabel("Frequency")
            axes[1].set_title("Residual Distribution")

            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"Could not load best model: {e}")
else:
    print("Processed data not available. Run preprocessing first.")

## 4. Best Model Learning Curve

Learning curve showing how the best model's performance changes with training size.

In [ ]:
if data is not None:
    best_run = df_runs.sort_values("val_score", ascending=False).iloc[0]
    model_uri = f"runs:/{best_run['run_id']}/model"

    try:
        best_model = mlflow.sklearn.load_model(model_uri)
        X_train = data["X_train"]
        y_train = data["y_train"]

        scoring = "f1_weighted" if task_type == "classification" else "neg_root_mean_squared_error"

        train_sizes, train_scores, val_scores = learning_curve(
            best_model, X_train, y_train,
            train_sizes=np.linspace(0.1, 1.0, 10),
            cv=5, scoring=scoring, n_jobs=-1,
        )

        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        val_mean = np.mean(val_scores, axis=1)
        val_std = np.std(val_scores, axis=1)

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(train_sizes, train_mean, "o-", color="steelblue",
                label="Training score", linewidth=2)
        ax.fill_between(train_sizes, train_mean - train_std,
                        train_mean + train_std, alpha=0.1, color="steelblue")
        ax.plot(train_sizes, val_mean, "o-", color="coral",
                label="Cross-validation score", linewidth=2)
        ax.fill_between(train_sizes, val_mean - val_std,
                        val_mean + val_std, alpha=0.1, color="coral")
        ax.set_xlabel("Training Set Size")
        ax.set_ylabel("Score")
        ax.set_title(f"Learning Curve ({best_run['model_name']})")
        ax.legend(loc="best")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Could not generate learning curve: {e}")
else:
    print("Processed data not available.")

## 5. Cross-Validation Score Distribution

Box plot of CV scores across experiments.

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(df_runs) * 1.2), 6))

# Create box plot data from cv_mean and cv_std
models = []
cv_means = []
cv_stds = []
labels = []

for _, row in df_runs.iterrows():
    if pd.notna(row["cv_mean"]) and pd.notna(row["cv_std"]):
        models.append(row["model_name"])
        cv_means.append(row["cv_mean"])
        cv_stds.append(row["cv_std"])
        labels.append(f"#{row['experiment_num']}\n{row['model_name'][:15]}")

x = range(len(cv_means))
ax.bar(x, cv_means, yerr=cv_stds, capsize=5,
       color="steelblue", alpha=0.8, edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("CV Score (mean +/- std)")
ax.set_title("Cross-Validation Score Distribution Across Experiments")
plt.tight_layout()
plt.show()

## 6. Summary Table

Top N models with their hyperparameters and metrics.

In [ ]:
# Summary table of top N models
summary_cols = ["experiment_num", "model_name", "val_score", "cv_mean",
                "cv_std", "training_time", "agent_notes"]

# Get parameter columns
param_cols = [c for c in df_runs.columns if c.startswith("param_") and
              c not in ["param_model_name", "param_agent_notes"]]

top_n_df = (df_runs
            .sort_values("val_score", ascending=False)
            .head(TOP_N)
            .reset_index(drop=True))

print(f"Top {TOP_N} Models")
print("=" * 80)

for i, row in top_n_df.iterrows():
    print(f"\nRank {i+1}: {row['model_name']}")
    print(f"  Experiment #:    {row['experiment_num']}")
    print(f"  Val Score:       {row['val_score']:.6f}")
    print(f"  CV Mean (+/-):   {row['cv_mean']:.6f} (+/- {row['cv_std']:.6f})")
    print(f"  Training Time:   {row['training_time']:.2f}s")
    print(f"  Notes:           {row['agent_notes']}")

    # Print key hyperparameters
    key_params = {c.replace('param_', ''): row[c]
                  for c in param_cols if pd.notna(row.get(c))}
    if key_params:
        print(f"  Key Params:      {key_params}")
    print("-" * 40)

# Also display as a clean dataframe
display_cols = [c for c in summary_cols if c in top_n_df.columns]
top_n_df[display_cols]